# CIFAR-10 Image Classification with Transfer Learning

## Computer Vision Capstone Mini-Project

This notebook completes an end-to-end image-classification workflow on a balanced, offline CIFAR-10 subset. It compares a CNN trained from scratch with MobileNetV2 feature extraction and fine-tuning, then evaluates the best model with per-class metrics, a confusion matrix, Grad-CAM, and a numbers-backed reflection.

**Deliverable name:** `cifar10_transfer_learning_image_classifier.ipynb`

## Prerequisites: Load the offline CIFAR-10 subset

The hosted lab is expected to have CIFAR-10 pre-seeded in `~/.keras/datasets`. This cell does not intentionally fetch a URL. If the cache is missing and `load_data()` attempts a network download, stop and contact the instructor.

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)

RNG = 42
np.random.seed(RNG)
tf.random.set_seed(RNG)

(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.cifar10.load_data()

CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]

def subset(x, y, per_class):
    y = y.ravel()
    rng = np.random.RandomState(RNG)
    idx = np.hstack([
        rng.choice(np.where(y == c)[0], per_class, replace=False)
        for c in range(10)
    ])
    rng.shuffle(idx)
    return x[idx].astype("float32") / 255.0, y[idx].astype("int32")

x_train, y_train = subset(x_train_full, y_train_full, 500)
x_test, y_test = subset(x_test_full, y_test_full, 100)

print(x_train.shape, x_test.shape, "range", x_train.min(), x_train.max())
assert x_train.shape == (5000, 32, 32, 3)
assert x_test.shape == (1000, 32, 32, 3)
assert np.isclose(x_train.min(), 0.0) and np.isclose(x_train.max(), 1.0)

## Step 1: Visualize and audit the data

The grid confirms image-label alignment. The count audit verifies exactly 500 training examples per class. CIFAR-10 is low resolution, so visually similar animal classes are expected to remain difficult.

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(9, 4.7))
for ax, i in zip(axes.ravel(), range(18)):
    ax.imshow(x_train[i])
    ax.set_title(CLASSES[y_train[i]], fontsize=8)
    ax.axis("off")
plt.suptitle("CIFAR-10 subset - sample images")
plt.tight_layout()
plt.show()

counts = np.bincount(y_train, minlength=10)
for c, n in enumerate(counts):
    print(f"{CLASSES[c]:12s} {n}")
assert np.all(counts == 500)

## Step 2: Baseline CNN trained from scratch

This model establishes the number transfer learning must beat. Early stopping restores the weights from the epoch with the best validation accuracy.

In [ ]:
def make_baseline_cnn():
    model = keras.Sequential([
        keras.Input((32, 32, 3)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax"),
    ], name="baseline_cnn")
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

baseline = make_baseline_cnn()
es = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=4, restore_best_weights=True
)
hist_base = baseline.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=25,
    batch_size=64,
    callbacks=[es],
    verbose=2,
)
base_test = baseline.evaluate(x_test, y_test, verbose=0)[1]
print(f"
BASELINE test accuracy = {base_test:.3f}")

### Baseline learning curves

A widening train-versus-validation gap suggests overfitting. Early stopping prevents continued training after validation accuracy stops improving.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(hist_base.history["accuracy"], label="train")
axes[0].plot(hist_base.history["val_accuracy"], label="validation")
axes[0].set_title("Baseline accuracy")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[1].plot(hist_base.history["loss"], label="train")
axes[1].plot(hist_base.history["val_loss"], label="validation")
axes[1].set_title("Baseline loss")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

## Step 3: Data augmentation

Horizontal flips, small rotations, and small zooms add realistic training variation. Augmentation runs only during training.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.06),
    layers.RandomZoom(0.1),
], name="augment")

img = x_train[1][None, ...]
fig, axes = plt.subplots(1, 5, figsize=(11, 2.6))
axes[0].imshow(img[0])
axes[0].set_title("original", fontsize=9)
axes[0].axis("off")
for ax in axes[1:]:
    ax.imshow(np.clip(data_augmentation(img, training=True)[0], 0, 1))
    ax.set_title("augmented", fontsize=9)
    ax.axis("off")
plt.suptitle(f"Augmentation preview - {CLASSES[y_train[1]]}")
plt.tight_layout()
plt.show()

## Step 4: Transfer learning, part A - feature extraction

MobileNetV2 expects larger inputs and its own preprocessing. The graph resizes 32×32 images to 96×96, converts `[0,1]` values to `[0,255]`, then applies `preprocess_input`. The ImageNet backbone is frozen while the new ten-class head trains.

If ImageNet weights are not already cached, `weights="imagenet"` may attempt a network download. In the offline lab, stop and contact the instructor rather than using an unapproved proxy.

In [ ]:
IMG = 96
base = keras.applications.MobileNetV2(
    input_shape=(IMG, IMG, 3),
    include_top=False,
    weights="imagenet",
)
base.trainable = False

inputs = keras.Input((32, 32, 3))
x = data_augmentation(inputs)
x = layers.Resizing(IMG, IMG)(x)
x = layers.Rescaling(255.0)(x)
x = keras.applications.mobilenet_v2.preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation="softmax")(x)
tl_model = keras.Model(inputs, outputs, name="mobilenetv2_cifar10")

tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
trainable_params = sum(np.prod(w.shape) for w in tl_model.trainable_weights)
print(f"trainable params (head only): {trainable_params:,}")

checkpoint_path = "cifar10_transfer_best.keras"
ckpt = keras.callbacks.ModelCheckpoint(
    checkpoint_path,
    monitor="val_accuracy",
    save_best_only=True,
)
es2 = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=3, restore_best_weights=True
)
hist_fe = tl_model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=8,
    batch_size=64,
    callbacks=[es2, ckpt],
    verbose=2,
)
fe_test = tl_model.evaluate(x_test, y_test, verbose=0)[1]
print(f"
FEATURE-EXTRACTION test accuracy = {fe_test:.3f}")

## Step 5: Transfer learning, part B - fine-tuning

Only the final 30 MobileNetV2 layers are unfrozen. A learning rate of `1e-5`, 100 times smaller than feature extraction, reduces the risk of destroying useful pretrained features.

In [ ]:
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

# Keep BatchNormalization layers frozen for stable fine-tuning on a small dataset.
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
es3 = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=3, restore_best_weights=True
)
hist_ft = tl_model.fit(
    x_train, y_train,
    validation_split=0.2,
    epochs=6,
    batch_size=64,
    callbacks=[es3, ckpt],
    verbose=2,
)
ft_test = tl_model.evaluate(x_test, y_test, verbose=0)[1]
print(f"
FINE-TUNED test accuracy = {ft_test:.3f}")
print(f"baseline={base_test:.3f}  feat-extract={fe_test:.3f}  fine-tuned={ft_test:.3f}")

### Transfer-learning curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(hist_fe.history["val_accuracy"], label="feature extraction")
axes[0].plot(hist_ft.history["val_accuracy"], label="fine-tuning")
axes[0].set_title("Validation accuracy")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[1].plot(hist_fe.history["val_loss"], label="feature extraction")
axes[1].plot(hist_ft.history["val_loss"], label="fine-tuning")
axes[1].set_title("Validation loss")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

## Step 6: Confusion matrix and per-class evaluation

This cell identifies the weakest class and the most frequent off-diagonal confusion pair for the final reflection.

In [ ]:
y_prob = tl_model.predict(x_test, verbose=0)
y_pred = y_prob.argmax(axis=1)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(7, 6.5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
    ax=ax, xticks_rotation=45, colorbar=False
)
plt.title("Confusion matrix - fine-tuned MobileNetV2")
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=CLASSES, digits=3))

precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, labels=np.arange(10), zero_division=0
)
worst_class_index = int(np.argmin(recall))
cm_without_diagonal = cm.copy()
np.fill_diagonal(cm_without_diagonal, 0)
true_index, predicted_index = np.unravel_index(
    np.argmax(cm_without_diagonal), cm_without_diagonal.shape
)
worst_pair_count = int(cm_without_diagonal[true_index, predicted_index])

print("Worst recall class:", CLASSES[worst_class_index], f"({recall[worst_class_index]:.3f})")
print(
    "Most frequent confusion:",
    f"true {CLASSES[true_index]} -> predicted {CLASSES[predicted_index]}",
    f"({worst_pair_count} images)",
)

## Step 7: Grad-CAM interpretation

The model is split into a preprocessing path, the MobileNetV2 convolutional backbone, and the classification head. Grad-CAM computes gradients of the selected class score with respect to the final convolutional feature maps.

In [ ]:
# Reconstruct a gradient model that exposes the backbone feature map.
preprocess_model = keras.Model(
    tl_model.input,
    base.input,
    name="gradcam_preprocess",
)
head_input = keras.Input(shape=base.output.shape[1:])
head_output = tl_model.layers[-3](head_input)
head_output = tl_model.layers[-2](head_output, training=False)
head_output = tl_model.layers[-1](head_output)
head_model = keras.Model(head_input, head_output, name="gradcam_head")

def gradcam(image, pred_index=None):
    processed = preprocess_model(image[None, ...], training=False)
    with tf.GradientTape() as tape:
        conv_out = base(processed, training=False)
        tape.watch(conv_out)
        predictions = head_model(conv_out, training=False)
        if pred_index is None:
            pred_index = int(tf.argmax(predictions[0]))
        score = predictions[:, pred_index]
    gradients = tape.gradient(score, conv_out)
    weights = tf.reduce_mean(gradients, axis=(0, 1, 2))
    cam = tf.maximum(tf.reduce_sum(conv_out[0] * weights, axis=-1), 0)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    return cam.numpy(), pred_index

correct = np.where(y_pred == y_test)[0]
show = correct[:3]
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for col, i in enumerate(show):
    cam, pidx = gradcam(x_test[i])
    cam_up = tf.image.resize(cam[..., None], (32, 32)).numpy().squeeze()
    axes[0, col].imshow(x_test[i])
    axes[0, col].axis("off")
    axes[0, col].set_title(f"truth/pred: {CLASSES[pidx]}", fontsize=9)
    axes[1, col].imshow(x_test[i])
    axes[1, col].imshow(cam_up, cmap="jet", alpha=0.45)
    axes[1, col].axis("off")
    axes[1, col].set_title("Grad-CAM", fontsize=9)
plt.suptitle("Where the fine-tuned model looks")
plt.tight_layout()
plt.show()

## Step 8: Numbers-backed written reflection

The following cell generates a 4–6 sentence reflection from the metrics produced by this notebook. Review the Grad-CAM images and replace the `gradcam_observation` text with what the heatmaps actually show before submission.

In [ ]:
print("=== RESULTS SUMMARY ===")
print(f"{'approach':22s}{'test acc':>10s}")
for name, accuracy in [
    ("Baseline CNN (scratch)", base_test),
    ("Transfer: feat-extract", fe_test),
    ("Transfer: fine-tuned", ft_test),
]:
    print(f"{name:22s}{accuracy:10.3f}")

absolute_lift = ft_test - base_test
feature_lift = fe_test - base_test
fine_tune_gain = ft_test - fe_test
print(f"
absolute lift (fine-tuned - baseline) = {absolute_lift:+.3f}")

gradcam_observation = (
    "The Grad-CAM overlays mainly concentrated on the central object regions; "
    "replace this sentence if your rendered heatmaps show background reliance."
)
reflection = f"""
The from-scratch CNN achieved {base_test:.3f} test accuracy, while the fine-tuned MobileNetV2 achieved {ft_test:.3f}, an absolute lift of {absolute_lift:+.3f}. Feature extraction contributed {feature_lift:+.3f} over the baseline because ImageNet pretraining supplied reusable edge, texture, and shape features despite having only 5,000 training images. Fine-tuning changed accuracy by {fine_tune_gain:+.3f} beyond feature extraction, showing whether gentle adaptation of the top backbone layers helped this CIFAR-10 domain. The weakest-recall class was {CLASSES[worst_class_index]} at {recall[worst_class_index]:.3f}, and the most frequent confusion was true {CLASSES[true_index]} predicted as {CLASSES[predicted_index]} for {worst_pair_count} test images. {gradcam_observation} A sensible next step is stronger but class-preserving augmentation and a controlled comparison with a CIFAR-friendly backbone or a larger balanced training subset.
""".strip()
print("
=== REFLECTION ===")
print(reflection)
print("Reflection sentence count target: 4-6 sentences")

# Save the finished model artifact.
tl_model.save("cifar10_transfer_learning_best.keras")
print("Saved best model -> cifar10_transfer_learning_best.keras")

## Submission checklist

- Run every cell from top to bottom.
- Confirm all plots are visible.
- Confirm baseline, feature-extraction, and fine-tuned test accuracies are printed.
- Confirm the confusion matrix and classification report are visible.
- Replace the Grad-CAM observation if it does not match the rendered heatmaps.
- Save the executed notebook as `cifar10_transfer_learning_image_classifier.ipynb`.